# 🍕 Food Image Classification System

This notebook demonstrates food image classification using Transfer Learning with MobileNetV2.

**Classes:** Pizza, Burger, Dosa, Biryani, Salad

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# Configuration
DATA_DIR = "dataset"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.0001

CLASS_NAMES = ['pizza', 'burger', 'dosa', 'biryani', 'salad']
NUM_CLASSES = len(CLASS_NAMES)

print(f"Classes: {CLASS_NAMES}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

## 3. Data Preprocessing & Augmentation

In [ ]:
# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# Validation data generator (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Create generators
train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    classes=CLASS_NAMES
)

validation_generator = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    classes=CLASS_NAMES
)

print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")

## 4. Visualize Sample Images

In [ ]:
# Get a batch of images
images, labels = next(train_generator)

# Plot sample images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    axes[i].imshow(images[i])
    class_idx = np.argmax(labels[i])
    axes[i].set_title(CLASS_NAMES[class_idx])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 5. Build the Model

In [ ]:
# Load pretrained MobileNetV2
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build complete model
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Train the Model

In [ ]:
# Callbacks
callbacks = [
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', 
                   save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=3, 
                 restore_best_weights=True, verbose=1)
]

# Train
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

## 7. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], 'b-', label='Training', linewidth=2)
axes[0].plot(history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], 'b-', label='Training', linewidth=2)
axes[1].plot(history.history['val_loss'], 'r-', label='Validation', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate Model

In [ ]:
# Evaluate
val_loss, val_acc = model.evaluate(validation_generator)
print(f"\nValidation Accuracy: {val_acc*100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")

## 9. Sample Predictions

In [ ]:
# Get validation images
images, true_labels = next(validation_generator)
predictions = model.predict(images[:8])

predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(true_labels[:8], axis=1)

# Plot
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i in range(8):
    axes[i].imshow(images[i])
    
    true_label = CLASS_NAMES[true_classes[i]]
    pred_label = CLASS_NAMES[predicted_classes[i]]
    confidence = predictions[i][predicted_classes[i]] * 100
    
    color = 'green' if predicted_classes[i] == true_classes[i] else 'red'
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label}\n{confidence:.1f}%", 
                     color=color, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 10. Save Model

In [ ]:
# Save model
model.save('food_classifier_notebook.keras')
print("Model saved!")

# Save class indices
import json
class_indices = {v: k for k, v in train_generator.class_indices.items()}
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)
print(f"Class indices: {class_indices}")

## 11. Test on Single Image

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_single_image(img_path):
    """Predict a single image."""
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    prediction = model.predict(img_array, verbose=0)
    predicted_class = CLASS_NAMES[np.argmax(prediction)]
    confidence = np.max(prediction) * 100
    
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence

# Example usage (update path to your image)
# predict_single_image('dataset/pizza/pizza1.jpg')